In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")


Using Colab cache for faster access to the 'elliptic-data-set' dataset.


In [ ]:
import torch

TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION = "cpu"

!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH_VERSION}+${CUDA_VERSION}.html
!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH_VERSION}+${CUDA_VERSION}.html
!pip install -q torch-cluster -f https://data.pyg.org/whl/torch-${TORCH_VERSION}+${CUDA_VERSION}.html
!pip install -q torch-spline-conv -f https://data.pyg.org/whl/torch-${TORCH_VERSION}+${CUDA_VERSION}.html
!pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.0/108.0 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.0/210.0 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.0 MB/s eta 0:00:00


In [ ]:
!wget -q https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_features.csv
!wget -q https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_classes.csv
!wget -q https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_edgelist.csv

!ls

sample_data


In [ ]:
%%writefile dataset.py

import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler


class EllipticDataset:
    def __init__(self, config):

        self.features_df = pd.read_csv(config.features_path, header=None)
        self.edges_df = pd.read_csv(config.edges_path)
        self.labels_df = pd.read_csv(config.classes_path)

        self.labels_df["class"] = self.labels_df["class"].map({
            "unknown": 2,
            "1": 1,
            "2": 0
        })

        self.merged_df = self.merge()

        self.edge_index = self._edge_index()
        self.edge_weights = self._edge_weights()
        self.node_features = self._node_features()
        self.labels = self._labels()

        self.train_mask, self.val_mask, self.test_mask = self._create_masks()

    def merge(self):
        df = self.features_df.merge(
            self.labels_df,
            how="left",
            right_on="txId",
            left_on=0
        )

        df = df.sort_values(0).reset_index(drop=True)
        return df

    def _node_features(self):

        node_features = self.merged_df.drop(['txId'], axis=1)
        node_features = node_features.drop(columns=[0, 1, "class"])

        scaler = StandardScaler()
        node_features = scaler.fit_transform(node_features.values)

        return torch.tensor(node_features, dtype=torch.float)

    def _labels(self):
        return torch.tensor(self.merged_df["class"].values, dtype=torch.long)

    def _edge_index(self):

        node_ids = self.merged_df[0].values
        id_map = {y: x for x, y in enumerate(node_ids)}

        edges = self.edges_df.copy()
        edges.txId1 = edges.txId1.map(id_map)
        edges.txId2 = edges.txId2.map(id_map)

        edges = edges.astype(int)

        edge_index = torch.tensor(edges.values.T, dtype=torch.long)

        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

        return edge_index.contiguous()

    def _edge_weights(self):
        return torch.ones(self.edge_index.shape[1], dtype=torch.float)

    def _create_masks(self):

        labels = self.labels.numpy()
        timesteps = self.merged_df[1].values

        train_mask = torch.zeros(len(labels), dtype=torch.bool)
        val_mask = torch.zeros(len(labels), dtype=torch.bool)
        test_mask = torch.zeros(len(labels), dtype=torch.bool)

        classified = labels != 2

        train_mask[(timesteps < 35) & classified] = True
        val_mask[(timesteps >= 35) & (timesteps < 40) & classified] = True
        test_mask[(timesteps >= 40) & classified] = True

        return train_mask, val_mask, test_mask

    def pyg_dataset(self):

        return Data(
            x=self.node_features,
            edge_index=self.edge_index,
            edge_attr=self.edge_weights,
            y=self.labels,
            train_mask=self.train_mask,
            val_mask=self.val_mask,
            test_mask=self.test_mask
        )

Writing dataset.py


In [ ]:
!ls
!wget https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_features.csv
!wget https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_classes.csv
!wget https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_edgelist.csv

dataset.py  sample_data
--2026-03-10 10:45:48--  https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_features.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-10 10:45:48 ERROR 404: Not Found.

--2026-03-10 10:45:48--  https://raw.githubusercontent.com/elliptic-graph-dataset/elliptic-graph-dataset.github.io/master/elliptic_txs_classes.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-03-10 10:45:48 ERROR 404: Not Found.

--2026-03-10 10:45:48--  https://raw.

In [ ]:
!mv elliptic_txs_features.csv features.csv
!mv elliptic_txs_classes.csv classes.csv
!mv elliptic_txs_edgelist.csv edgelist.csv

mv: cannot stat 'elliptic_txs_features.csv': No such file or directory
mv: cannot stat 'elliptic_txs_classes.csv': No such file or directory
mv: cannot stat 'elliptic_txs_edgelist.csv': No such file or directory


In [ ]:
!ls features.csv
!ls edgelist.csv
!ls classes.csv

ls: cannot access 'features.csv': No such file or directory
ls: cannot access 'edgelist.csv': No such file or directory
ls: cannot access 'classes.csv': No such file or directory


In [ ]:
%%writefile model.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


class GCN(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.conv1 = GCNConv(config.input_dim, config.hidden_dim)
        self.conv2 = GCNConv(config.hidden_dim, config.output_dim)

        self.residual = nn.Linear(config.input_dim, config.output_dim)

        self.dropout = config.dropout

    def forward(self, data):

        x, edge_index = data.x, data.edge_index

        h = self.conv1(x, edge_index)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = self.conv2(h, edge_index)

        return h + self.residual(x)

Writing model.py


In [ ]:
import sys
sys.path.append("/content")

In [ ]:
%%writefile dataset.py
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler


class EllipticDataset:
    def __init__(self, config):
        self.features_df = pd.read_csv(config.features_path, header=None)
        self.edges_df = pd.read_csv(config.edges_path)
        self.labels_df = pd.read_csv(config.classes_path)

        # Map classes
        # licit = 0, illicit = 1, unknown = 2
        self.labels_df["class"] = self.labels_df["class"].map({'unknown': 2, '1': 1, '2': 0})

        self.merged_df = self.merge()

        self.edge_index = self._edge_index()
        self.edge_weights = self._edge_weights()
        self.node_features = self._node_features()
        self.labels = self._labels()

        self.train_mask, self.val_mask, self.test_mask = self._create_masks()

    # ------------------------------------------------

    def merge(self):
        df_merge = self.features_df.merge(
            self.labels_df, how='left', right_on="txId", left_on=0
        )
        df_merge = df_merge.sort_values(0).reset_index(drop=True)
        return df_merge

    # ------------------------------------------------

    def _node_features(self):
        node_features = self.merged_df.drop(['txId'], axis=1).copy()
        node_features = node_features.drop(columns=[0, 1, "class"])

        # 🔥 Feature Normalization (VERY IMPORTANT)
        scaler = StandardScaler()
        node_features = scaler.fit_transform(node_features.values)

        return torch.tensor(node_features, dtype=torch.float)

    # ------------------------------------------------

    def _labels(self):
        labels = self.merged_df["class"].values
        return torch.tensor(labels, dtype=torch.long)

    # ------------------------------------------------

    def _edge_index(self):
        node_ids = self.merged_df[0].values
        ids_mapping = {y: x for x, y in enumerate(node_ids)}

        edges = self.edges_df.copy()
        edges.txId1 = edges.txId1.map(ids_mapping)
        edges.txId2 = edges.txId2.map(ids_mapping)
        edges = edges.astype(int)

        edge_index = torch.tensor(edges.values.T, dtype=torch.long)

        # 🔥 Make graph undirected
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

        return edge_index.contiguous()

    # ------------------------------------------------

    def _edge_weights(self):
        return torch.ones(self.edge_index.shape[1], dtype=torch.float)

    # ------------------------------------------------

    def _create_masks(self):
        labels = self.labels.numpy()
        timesteps = self.merged_df[1].values  # timestep column

        train_mask = torch.zeros(len(labels), dtype=torch.bool)
        val_mask = torch.zeros(len(labels), dtype=torch.bool)
        test_mask = torch.zeros(len(labels), dtype=torch.bool)

        # Only classified nodes (ignore unknown)
        classified = labels != 2

        # 🔥 Temporal split (proper for Elliptic)
        train_mask[(timesteps < 35) & classified] = True
        val_mask[(timesteps >= 35) & (timesteps < 40) & classified] = True
        test_mask[(timesteps >= 40) & classified] = True

        return train_mask, val_mask, test_mask

    # ------------------------------------------------

    def get_class_weights(self):
        labels = self.labels[self.train_mask]
        class_sample_count = torch.bincount(labels[labels != 2])
        weight = 1. / class_sample_count.float()
        return weight

    # ------------------------------------------------

    def pyg_dataset(self):
        data = Data(
            x=self.node_features,
            edge_index=self.edge_index,
            edge_attr=self.edge_weights,
            y=self.labels,
            train_mask=self.train_mask,
            val_mask=self.val_mask,
            test_mask=self.test_mask
        )
        return data

Overwriting dataset.py


In [ ]:
%%writefile trainer.py

import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


class Trainer:

    def __init__(self, model, dataset, config):

        self.model = model
        self.data = dataset.pyg_dataset().to(config.device)
        self.config = config

        self.model = self.model.to(config.device)

        class_weights = torch.tensor([1.0, 1.4]).to(config.device)

        self.criterion = nn.CrossEntropyLoss(weight=class_weights)

        self.optimizer = optim.Adam(
            self.model.parameters(),
            lr=config.lr,
            weight_decay=config.weight_decay
        )

    def train(self):

        print("Starting training...\n")

        for epoch in range(self.config.epochs):

            self.model.train()
            self.optimizer.zero_grad()

            out = self.model(self.data)

            mask = self.data.train_mask & (self.data.y != 2)

            loss = self.criterion(out[mask], self.data.y[mask])

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
            self.optimizer.step()

            if epoch % 10 == 0:
                print(f"Epoch {epoch} | Loss {loss.item():.4f}")
                self.evaluate()
                print("-"*40)

        print("Training finished\n")
        self.evaluate()

    @torch.no_grad()
    def evaluate(self):

        self.model.eval()

        out = self.model(self.data)
        preds = out.argmax(dim=1)

        self._metrics("Train", self.data.train_mask, preds)
        self._metrics("Val", self.data.val_mask, preds)
        self._metrics("Test", self.data.test_mask, preds)

    def _metrics(self, name, mask, preds):

        mask = mask & (self.data.y != 2)

        y_true = self.data.y[mask].cpu()
        y_pred = preds[mask].cpu()

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        print(f"{name} → Acc:{acc:.3f} Prec:{prec:.3f} Rec:{rec:.3f} F1:{f1:.3f}")

Writing trainer.py


In [ ]:
import kagglehub
import os

# Download latest version of the dataset to ensure it's available
path = kagglehub.dataset_download("ellipticco/elliptic-data-set")

# Verify the contents of the downloaded directory
print(f"Dataset downloaded to: {path}")
print("Contents of the dataset directory:")
!ls -F {path}

100%|██████████| 146M/146M [00:00<00:00, 207MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/ellipticco/elliptic-data-set/versions/1
Contents of the dataset directory:
elliptic_bitcoin_dataset/


In [ ]:
import torch
from model import GCN
from trainer import Trainer
import importlib
import sys


class Config:

    features_path = "/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_features.csv"
    edges_path = "/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_edgelist.csv"
    classes_path = "/kaggle/input/elliptic-data-set/elliptic_bitcoin_dataset/elliptic_txs_classes.csv"

    input_dim = 165
    hidden_dim = 64
    output_dim = 2
    dropout = 0.5

    lr = 0.004
    weight_decay = 5e-4
    epochs = 60

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


config = Config()

# Reload the dataset module to ensure the latest changes are picked up
if 'dataset' in sys.modules:
    importlib.reload(sys.modules['dataset'])
from dataset import EllipticDataset

dataset = EllipticDataset(config)
model = GCN(config)

trainer = Trainer(model, dataset, config)
trainer.train()

Starting training...

Epoch 0 | Loss 0.7586
Train → Acc:0.760 Prec:0.266 Rec:0.609 F1:0.371
Val → Acc:0.736 Prec:0.129 Rec:0.389 F1:0.194
Test → Acc:0.808 Prec:0.089 Rec:0.258 F1:0.133
----------------------------------------
Epoch 10 | Loss 0.2143
Train → Acc:0.930 Prec:0.682 Rec:0.735 F1:0.708
Val → Acc:0.852 Prec:0.339 Rec:0.868 F1:0.488
Test → Acc:0.814 Prec:0.168 Rec:0.572 F1:0.259
----------------------------------------
Epoch 20 | Loss 0.1880
Train → Acc:0.941 Prec:0.721 Rec:0.806 F1:0.761
Val → Acc:0.882 Prec:0.397 Rec:0.855 F1:0.542
Test → Acc:0.855 Prec:0.203 Rec:0.533 F1:0.294
----------------------------------------
Epoch 30 | Loss 0.1620
Train → Acc:0.953 Prec:0.796 Rec:0.804 F1:0.800
Val → Acc:0.915 Prec:0.488 Rec:0.850 F1:0.620
Test → Acc:0.897 Prec:0.283 Rec:0.530 F1:0.369
----------------------------------------
Epoch 40 | Loss 0.1446
Train → Acc:0.959 Prec:0.812 Rec:0.839 F1:0.825
Val → Acc:0.921 Prec:0.508 Rec:0.839 F1:0.633
Test → Acc:0.924 Prec:0.380 Rec:0.519 F1:0